# Emotion-Based Talking Head Generation (Standalone)

Generates **emotion-controlled talking head videos** with IMTalker using an **already-trained
emotion LoRA adapter** downloaded from Hugging Face:
[`bitwisemind/emotion_lora_adapter_v1`](https://huggingface.co/bitwisemind/emotion_lora_adapter_v1)
(file `last.ckpt`).

**This repository is fully self-contained** -- all required source code (`generator/`,
`renderer/`) is bundled, so no external repository is needed. It reuses the exact avatar
testing / video generation method that runs after each training epoch in
`05_train_emotion_lora.ipynb` from the full training pipeline -- but **without any LoRA
training** and without the real-time PersonaPlex conversation pipeline.

**Quick start on RunPod**
1. `git clone https://github.com/MoshiHead/emotion_talking_head_test_only.git`
2. `cd emotion_talking_head_test_only` and open this notebook
3. Run all cells -- models download automatically, then the video is generated

**Pipeline**
1. Verify the bundled source code and locate the repository root
2. Set up the environment (CUDA PyTorch for your GPU + dependencies)
3. Download and configure all required models:
   - Base IMTalker generator / renderer / wav2vec2 (`cbsjtu01/IMTalker`)
   - Style LoRA merged into the base generator (`niloy629/hdtf_preprocess`) -- matches the
     emotion adapter training setup
   - Emotion LoRA adapter `last.ckpt` (`bitwisemind/emotion_lora_adapter_v1`)
4. Load the avatar: base generator -> style LoRA merge -> emotion LoRA adapters -> emotion checkpoint
5. Generate a talking head video for the configured reference image, audio, and emotion

**Requirements**
- Any CUDA GPU (RTX 3090/4090/5090, A100, A40, A6000, L40S, H100, ...)
- `ffmpeg` available on `PATH` (preinstalled on RunPod images)
- ~2 GB of disk space for model downloads

**Emotions**: `neutral` (0), `happy` (1), `sad` (2), `angry` (3), `surprise` (4), `fear` (5)

In [ ]:
# ============================================================================
# 1. Locate the repository root (the folder containing generator/ and renderer/)
# ============================================================================
from pathlib import Path


def find_repo_root():
    """Walk up from the current working directory to find the repo root."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "generator" / "FM.py").exists() and (cand / "renderer" / "models.py").exists():
            return cand
    raise FileNotFoundError(
        "Could not locate the repository root. Run this notebook from inside the "
        "cloned repository folder (the folder that contains generator/ and renderer/)."
    )


REPO_ROOT = find_repo_root()

required_files = [
    "generator/FM.py",
    "generator/FMT.py",
    "generator/wav2vec2.py",
    "generator/generate.py",
    "generator/train_lora.py",
    "generator/options/base_options.py",
    "renderer/models.py",
    "renderer/modules.py",
    "renderer/attention_modules.py",
    "renderer/lia_resblocks.py",
]
for rel in required_files:
    assert (REPO_ROOT / rel).exists(), f"Missing required file: {rel}"

print(f"Repository root: {REPO_ROOT}")
print(f"Verified {len(required_files)} required source files")
print("Code structure OK -- no external repository needed")

In [ ]:
# ============================================================================
# 2. CONFIGURATION -- EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Inputs (EDIT ME)
# ---------------------------------------------------------------------------
# Reference image: portrait / avatar face (resized to 512x512)
REF_IMAGE = str(REPO_ROOT / "assets" / "3robert.jpeg")

# Input audio: .wav speech file (16 kHz mono recommended)
AUDIO = str(REPO_ROOT / "assets" / "audio_3.wav")

# Emotion name: "neutral", "happy", "sad", "angry", "surprise", "fear"
# (also accepts an integer 0-5, or "all" to render all 6 emotions)
EMOTION = "happy"

# ---------------------------------------------------------------------------
# Generation settings (same defaults as the per-epoch training tests)
# ---------------------------------------------------------------------------
CFG_SCALE = 3.0     # audio classifier-free guidance scale
NFE       = 10      # number of flow-matching function evaluations
SEED      = 42      # random seed
CROP      = False   # False = plain 512x512 resize (safe for robot faces, no face detection)

# Where generated videos are saved
OUTPUT_DIR = str(REPO_ROOT / "emotion_talking_head_outputs")

# ---------------------------------------------------------------------------
# Emotion LoRA adapter (downloaded from Hugging Face)
# ---------------------------------------------------------------------------
HF_EMOTION_LORA_REPO = "bitwisemind/emotion_lora_adapter_v1"
HF_EMOTION_LORA_FILE = "last.ckpt"
EMOTION_LORA_RANK  = 32     # must match training (LORA_RANK in 05_train_emotion_lora.ipynb)
EMOTION_LORA_ALPHA = 64.0   # must match training (LORA_ALPHA)

# ---------------------------------------------------------------------------
# Optional style LoRA merged into the base generator before the emotion
# adapter is applied (matches how the emotion adapter was trained)
# ---------------------------------------------------------------------------
USE_STYLE_LORA     = True
HF_STYLE_LORA_REPO = "niloy629/hdtf_preprocess"    # dataset repository
HF_STYLE_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
STYLE_LORA_RANK    = 64
STYLE_LORA_ALPHA   = 128.0

# ---------------------------------------------------------------------------
# Derived paths (no need to edit) -- all downloads stay inside this repo folder
# ---------------------------------------------------------------------------
CKPT_DIR            = REPO_ROOT / "checkpoints"
WAV2VEC_DIR         = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT       = CKPT_DIR / "renderer.ckpt"
BASE_GENERATOR_CKPT = CKPT_DIR / "generator.ckpt"
STYLE_LORA_CKPT     = CKPT_DIR / HF_STYLE_LORA_FILE
EMOTION_LORA_CKPT   = CKPT_DIR / "emotion_lora" / HF_EMOTION_LORA_FILE

EMOTION_MAP   = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}


def resolve_emotions(spec):
    """Accepts a name ('happy'), an integer (1), or 'all'."""
    if isinstance(spec, int):
        return [spec]
    s = str(spec).strip().lower()
    if s == "all":
        return [0, 1, 2, 3, 4, 5]
    if s.isdigit():
        return [int(s)]
    assert s in EMOTION_MAP, f"Unknown emotion {spec!r}. Valid: {list(EMOTION_MAP)} or 'all'"
    return [EMOTION_MAP[s]]


EMOTION_IDS = resolve_emotions(EMOTION)

print(f"reference image : {REF_IMAGE}")
print(f"audio           : {AUDIO}")
print(f"emotion(s)      : {[EMOTION_NAMES[i] for i in EMOTION_IDS]} (ids={EMOTION_IDS})")
print(f"generation      : cfg={CFG_SCALE}  nfe={NFE}  seed={SEED}  crop={CROP}")
print(f"output dir      : {OUTPUT_DIR}")
print(f"emotion adapter : {HF_EMOTION_LORA_REPO}/{HF_EMOTION_LORA_FILE} "
      f"(rank={EMOTION_LORA_RANK}, alpha={EMOTION_LORA_ALPHA})")
print(f"style LoRA      : {HF_STYLE_LORA_FILE if USE_STYLE_LORA else '(disabled)'}")

In [ ]:
# ============================================================================
# 3. Environment setup -- CUDA PyTorch + dependencies
#    Works on any RunPod GPU (RTX 30/40/50 series, A100, A40, A6000, L40S, H100)
# ============================================================================
import importlib.util, shutil, subprocess, sys


def _pip(*args):
    print("[install]", " ".join(a for a in args if not a.startswith("http")))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(args), check=True)


# --- 1. Make sure a CUDA-capable PyTorch build is installed ------------------
def torch_cuda_ok():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


if not torch_cuda_ok():
    print("CUDA-capable PyTorch not detected -- installing ...")
    # cu128 wheels cover sm_75..sm_120 (RTX 3090/4090/5090, A100, A40, A6000, L40S, H100).
    # cu126 is the fallback, then default PyPI wheels.
    installed = False
    for index_url in [
        "https://download.pytorch.org/whl/cu128",
        "https://download.pytorch.org/whl/cu126",
    ]:
        try:
            print(f"[install] torch + torchvision from {index_url.rsplit('/', 1)[-1]} wheels ...")
            _pip("torch", "torchvision", "--index-url", index_url)
            if torch_cuda_ok():
                installed = True
                break
        except subprocess.CalledProcessError:
            print("  wheel index failed, trying next option ...")
    if not installed:
        _pip("torch", "torchvision")   # default PyPI wheels (latest CUDA)
        assert torch_cuda_ok(), "CUDA still unavailable after installing PyTorch."
else:
    print("CUDA-capable PyTorch already installed -- skipping reinstall")

import torch
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
print(f"torch:              {torch.__version__}")
print(f"GPU:                {gpu_name}")
print(f"Compute capability: {cc[0]}.{cc[1]}")
print(f"VRAM:               {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (required for video encoding)."

# --- 2. Python dependencies ---------------------------------------------------
required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",
    "torchdiffeq":       "torchdiffeq==0.2.5",
    "einops":            "einops",
    "transformers":      "transformers>=4.38,<4.42",
    "librosa":           "librosa",
    "face_alignment":    "face_alignment==1.4.1",
    "av":                "av==12.0.0",
    "tensorboard":       "tensorboard",
    "timm":              "timm==1.0.9",
    "tqdm":              "tqdm",
    "huggingface_hub":   "huggingface_hub",
    "yaml":              "pyyaml",
    "cv2":               "opencv-python<4.10.0.0",
    "matplotlib":        "matplotlib<3.9.0",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip(*missing)
else:
    print("All required packages already installed")
print("Environment OK")

In [ ]:
# ============================================================================
# 4. Download all required models & checkpoints
# ============================================================================
from huggingface_hub import hf_hub_download

CKPT_DIR.mkdir(parents=True, exist_ok=True)


def download(repo_id, filename, local_dir, repo_type="model"):
    dest = Path(local_dir) / filename
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {dest}")
        return dest
    print(f"[download] {filename} from {repo_id} ...")
    hf_hub_download(repo_id=repo_id, filename=filename,
                    local_dir=str(local_dir), repo_type=repo_type)
    print(f"    ready: {dest}")
    return dest


# 1) Base IMTalker models (config, renderer, full base generator)
for fname in ["config.yaml", "renderer.ckpt", "generator.ckpt"]:
    download("cbsjtu01/IMTalker", fname, CKPT_DIR)

# 2) wav2vec2-base-960h audio encoder
for rel in [
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]:
    download("cbsjtu01/IMTalker", rel, CKPT_DIR)

# 3) Style LoRA (merged into the base generator, matching emotion-adapter training)
if USE_STYLE_LORA:
    download(HF_STYLE_LORA_REPO, HF_STYLE_LORA_FILE,
             CKPT_DIR, repo_type="dataset")

# 4) Trained emotion LoRA adapter from Hugging Face
download(HF_EMOTION_LORA_REPO, HF_EMOTION_LORA_FILE, EMOTION_LORA_CKPT.parent)

# Final verification
for p in [RENDERER_CKPT, BASE_GENERATOR_CKPT,
          WAV2VEC_DIR / "pytorch_model.bin", EMOTION_LORA_CKPT]:
    assert Path(p).exists(), f"Missing required file: {p}"
if USE_STYLE_LORA:
    assert STYLE_LORA_CKPT.exists(), f"Missing style LoRA: {STYLE_LORA_CKPT}"

print("\nAll models ready:")
print(f"  base generator : {BASE_GENERATOR_CKPT}")
print(f"  renderer       : {RENDERER_CKPT}")
print(f"  wav2vec2       : {WAV2VEC_DIR}")
if USE_STYLE_LORA:
    print(f"  style LoRA     : {STYLE_LORA_CKPT}")
print(f"  emotion LoRA   : {EMOTION_LORA_CKPT}")

In [ ]:
# ============================================================================
# 5. Load the avatar model
#    (same method as the per-epoch test in 05_train_emotion_lora.ipynb,
#     i.e. generator/test_emotions.py -- also bundled in this repo)
# ============================================================================
import argparse, math, os, subprocess, sys, tempfile
import torch
import torch.nn.functional as F

# torch >= 2.6 changed the weights_only default; keep the old behaviour
_TORCH_LOAD = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _TORCH_LOAD(*args, **kwargs)
torch.load = _torch_load_compat

for p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if p not in sys.path:
        sys.path.insert(0, p)

from generator.options.base_options import BaseOptions
from generator.FM import FMGenerator, EMOTION_MAP, EMOTION_NAMES
from generator.train_lora import apply_emotion_lora_to_model, _get_child
from generator.generate import DataProcessor, InferenceOptions
from renderer.models import IMTRenderer
from generator.wav2vec2 import Wav2VecModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Generator -------------------------------------------------------------
parser = argparse.ArgumentParser()
parser = BaseOptions().initialize(parser)
parser.add_argument("--skip_fm_audio_encoder", action="store_true")
opt = parser.parse_args([])
opt.rank = device
opt.wav2vec_model_path = str(WAV2VEC_DIR)
opt.skip_fm_audio_encoder = True
opt.fix_noise_seed = True
opt.seed = SEED

model = FMGenerator(opt).to(device)

# 1) Load the full base generator weights (the 38.8M-parameter avatar model)
b_ckpt = torch.load(str(BASE_GENERATOR_CKPT), map_location="cpu")
b_raw = b_ckpt.get("ema_state_dict") or b_ckpt.get("state_dict", b_ckpt)
if "model" in b_raw and isinstance(b_raw["model"], dict):
    b_raw = b_raw["model"]
b_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v
           for k, v in b_raw.items()}
model.load_state_dict(b_state, strict=False)
print(f"Base generator loaded ({len(b_state)} keys)")

# 2) Merge the style LoRA into the base weights:  dW = (alpha/rank) * B @ A
if USE_STYLE_LORA and STYLE_LORA_CKPT.exists():
    s_ckpt = torch.load(str(STYLE_LORA_CKPT), map_location="cpu")
    s_raw = s_ckpt.get("ema_state_dict") or s_ckpt.get("state_dict", s_ckpt)
    if "model" in s_raw and isinstance(s_raw["model"], dict):
        s_raw = s_raw["model"]
    s_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v
               for k, v in s_raw.items()}
    lora_targets = {k.replace(".lora_A.weight", "") for k in s_state
                    if ".lora_A.weight" in k}
    style_scale = STYLE_LORA_ALPHA / STYLE_LORA_RANK
    for target in lora_targets:
        A = s_state.get(f"{target}.lora_A.weight")
        B = s_state.get(f"{target}.lora_B.weight")
        if A is not None and B is not None:
            mod = _get_child(model, target)
            mod.weight.data += (style_scale * (B @ A)).to(mod.weight.device)
    print(f"Merged {len(lora_targets)} style LoRA layers (scale={style_scale:.2f})")

# 3) Attach the emotion LoRA adapters + emotion conditioner
apply_emotion_lora_to_model(model, rank=EMOTION_LORA_RANK,
                            alpha=EMOTION_LORA_ALPHA, dropout=0.0)

# 4) Load the trained emotion adapter (last.ckpt from Hugging Face)
e_ckpt = torch.load(str(EMOTION_LORA_CKPT), map_location="cpu")
e_raw = e_ckpt.get("ema_state_dict") or e_ckpt.get("state_dict", e_ckpt)
if "model" in e_raw and isinstance(e_raw["model"], dict):
    e_raw = e_raw["model"]
e_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v
           for k, v in e_raw.items()}
missing, unexpected = model.load_state_dict(e_state, strict=False)
print(f"Emotion adapter loaded: {EMOTION_LORA_CKPT.name} "
      f"(missing={len(missing)}, unexpected={len(unexpected)})")
model.eval()

# --- Renderer ---------------------------------------------------------------
p_inf = argparse.ArgumentParser()
p_inf = InferenceOptions().initialize(p_inf)
opt_inf = p_inf.parse_args([])
opt_inf.rank = device
opt_inf.renderer_path = str(RENDERER_CKPT)
opt_inf.wav2vec_model_path = str(WAV2VEC_DIR)

renderer = IMTRenderer(opt_inf).to(device).eval()
r_ckpt = torch.load(str(RENDERER_CKPT), map_location="cpu")
r_state = {k.replace("gen.", ""): v for k, v in r_ckpt["state_dict"].items()
           if k.startswith("gen.")}
renderer.load_state_dict(r_state, strict=False)
print("Renderer loaded")

# --- Wav2Vec2 audio encoder ---------------------------------------------------
wav2vec = Wav2VecModel.from_pretrained(str(WAV2VEC_DIR), local_files_only=True).to(device).eval()
data_processor = DataProcessor(opt_inf)
print("Wav2Vec2 + DataProcessor ready")

In [ ]:
# ============================================================================
# 6. Preprocess the reference image and audio (same as the per-epoch test)
# ============================================================================
ref_path, aud_path = Path(REF_IMAGE), Path(AUDIO)
assert ref_path.exists(), f"Reference image not found: {ref_path}"
assert aud_path.exists(), f"Audio file not found: {aud_path}"
print(f"Reference image: {ref_path}")
print(f"Audio file:      {aud_path}")

data = data_processor.preprocess(str(ref_path), str(aud_path), crop=CROP)
s = data["s"].to(device)
f_r, g_r = renderer.dense_feature_encoder(s)
t_r = renderer.latent_token_encoder(s)
data["ref_x"] = t_r
data["pose"] = None
data["cam"] = None
data["gaze"] = None

# Wav2Vec features resampled onto the 25 fps video grid
raw_audio = data["a"].squeeze(0).to(device)
T_target = math.ceil(raw_audio.shape[0] * 25 / 16000)
target_samples = int(T_target * 16000 / 25)
if raw_audio.shape[0] < target_samples:
    diff = target_samples - raw_audio.shape[0]
    raw_audio = F.pad(raw_audio.unsqueeze(0), (0, diff), mode="replicate").squeeze(0)
else:
    raw_audio = raw_audio[:target_samples]
with torch.no_grad():
    out = wav2vec(raw_audio.unsqueeze(0), seq_len=T_target, output_hidden_states=False)
    data["a_feat"] = out.last_hidden_state

print(f"Prepared {T_target} video frames ({T_target / 25:.1f}s) - ready for generation")

In [ ]:
# ============================================================================
# 7. Generate the emotion-controlled talking head video(s)
# ============================================================================
from IPython.display import Video, display


def save_video(vid_tensor, video_path, audio_path):
    """Encode frames with torchvision, then mux the original audio with ffmpeg."""
    import torchvision
    video_path = Path(video_path)
    video_path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        temp_filename = tmp.name
    vid = vid_tensor.permute(0, 2, 3, 1).detach().clamp(0, 1).cpu()
    torchvision.io.write_video(temp_filename, (vid * 255).to(torch.uint8), fps=25)
    res = subprocess.run(
        f'ffmpeg -y -v warning -i "{temp_filename}" -i "{audio_path}" '
        f'-c:v copy -c:a aac -shortest "{video_path}"',
        shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        os.remove(temp_filename)
    else:
        print(f"    [warn] ffmpeg mux failed ({res.stderr.strip()[:200]}); "
              f"saving video without audio")
        import shutil
        shutil.move(temp_filename, str(video_path))


out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
generated = []

for eid in EMOTION_IDS:
    emo_name = EMOTION_NAMES.get(eid, f"emotion_{eid}")
    print(f"Rendering [{eid}] {emo_name} ...", flush=True)
    data["emotion_id"] = eid

    with torch.no_grad():
        # Flow-matching sampling with the emotion condition (same as per-epoch tests)
        sample = model.sample(data, a_cfg_scale=CFG_SCALE, nfe=NFE, seed=SEED)

        # Decode the motion latents into frames with the renderer
        T = sample.shape[1]
        ta_r = renderer.adapt(t_r, g_r)
        m_r = renderer.latent_token_decoder(ta_r)
        d_hat = []
        for t in range(T):
            ta_c = renderer.adapt(sample[:, t, ...], g_r)
            m_c = renderer.latent_token_decoder(ta_c)
            d_i = renderer.decode(m_c, m_r, f_r)
            d_hat.append(d_i)

        vid_tensor = torch.cat(d_hat, dim=0)   # [T, 3, 512, 512]
        out_file = out_dir / f"{eid}_{emo_name}.mp4"
        save_video(vid_tensor, out_file, str(aud_path))

    print(f"    saved -> {out_file}")
    generated.append(out_file)

print(f"\nDone! {len(generated)} video(s) saved in: {out_dir.resolve()}")
for f in generated:
    print(f"  {f.name}")

In [ ]:
# ============================================================================
# 8. Display the generated video(s) inline
# ============================================================================
for f in generated:
    print(f.name)
    display(Video(str(f), embed=True, width=512))

## Notes & Troubleshooting

- **Change emotion / inputs**: edit the CONFIGURATION cell (2), then re-run the *Preprocess* (6)
  and *Generate* (7) cells. Changing only the emotion requires re-running just cell 7.
- **Rank / alpha**: `EMOTION_LORA_RANK = 32` and `EMOTION_LORA_ALPHA = 64.0` must match the
  values used when the adapter was trained (`05_train_emotion_lora.ipynb`).
- **Robot / non-human faces**: keep `CROP = False` (plain 512x512 resize, no face-alignment
  cropping -- face detection fails on robot faces).
- **Black frames**: almost always means the full base `generator.ckpt` was not loaded
  correctly -- re-run the download (4) and model-loading (5) cells.
- **Style LoRA**: the emotion adapter was trained on top of the base generator with the style
  LoRA merged in; keep `USE_STYLE_LORA = True` to reproduce the training-time behaviour.
- **All emotions**: set `EMOTION = "all"` to render `0_neutral.mp4` ... `5_fear.mp4` in one run
  (same behaviour as `generator/test_emotions.py` without the `--emotions` filter).
- **GPU compatibility**: the environment cell (3) automatically installs the right PyTorch
  CUDA build for the detected GPU (cu128 covers RTX 30/40/50 series, A100, A40, L40S, H100;
  cu126 is the automatic fallback). On standard RunPod PyTorch images the preinstalled build
  is used as-is.
- **CLI alternative**: the bundled `generator/test_emotions.py` can also render videos from
  the terminal, e.g. `python generator/test_emotions.py --checkpoint checkpoints/emotion_lora/last.ckpt
  --ref_img assets/3robert.jpeg --audio assets/audio_3.wav --out_dir outputs --cfg 3.0`